## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [ ]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [ ]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [ ]:
file_path = os.path.join("data/input", "Selisih Dengan Lapus.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [ ]:
# Membaca data dari file excel (Selisih dgn Lapus)
df = read_data_from_excel(file_path, "Selisih dgn Lapus")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

#### 2. Evaluasi selisih ADHB dan ADHK

In [ ]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("selisih")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_selisih_lapus = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        if pd.notna(val) and (val > 0.05 or val < -0.05):
            # Pisahkan nama kolom jadi komponen
            # contoh: adhb_pdrb_q1-25 → ['adhb', 'pdrb', 'q1-25']
            parts = col.split("_")
            
            periode = parts[-1]  # ambil q1-25

            record = {
                "kode": df_clean.at[idx, "kode"],
                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                "periode": periode,
                "nilai": val,
                "kolom": col,  # tambahan biar tau sumbernya
                "evaluasi": "Terdapat selisih nilai PDRB antara Lapangan Usaha dan Pengeluaran"
            }
            evaluasi_selisih_lapus.append(record)
                    
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_selisih_lapus = pd.DataFrame(evaluasi_selisih_lapus)
print(evaluasi_selisih_lapus)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_selisih_lapus])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "kabupaten/kota")